# 🔗 lead_lag — does one stock's move predict another's *later* move?

The go/no-go exploration before any lead-lag / cross-asset-momentum / delayed-correlation strategy
(README). It covers **every leader × follower pair** of a broad universe: ~514 names at **1-minute
and 5-minute** bars and ~1,900 at **daily** bars. For each pair it measures whether the leader's
last few bars predict the follower's next one. Then it checks whether that survives the three
things that manufacture a lead-lag: **stale prints**, **the market**, and **luck** across hundreds
of thousands of pairs. Finally it tests the two usual catalysts: **size** (large caps react faster)
and **economic links** (industries, supply chains, metal → miners).

| § | question |
|---|---|
| 1 | the data — 1m (→ 5m) and daily cubes, point-in-time sizes, industries, how often each name prints |
| 2 | units, costs and **power** — what correlation would pay for costs, and can the map see it? |
| 3 | **does the market lead?** the fast market's past → everyone else (C1) |
| 4 | **the pair map** — every leader × follower beyond the market, against rolled nulls (V1, V2, C2) |
| 5 | **size and staleness** — who lags: small names, or names that rarely print? (C4) |
| 6 | **links** — industries, big → small inside industries, supply chains, metal → miners (C5) |
| 7 | **persistence** — do discovery's pairs and laggards hold on validation? (C3) — *the screen table* |
| 8 | **tradability** — entered at the next open, net of costs, on the validation half (C6) |
| 9 | verdict per bar size — GO / NO-GO, checks, scorecard |

**Reading a T.** Every pair statistic is a *session-clustered t* (weeks at 1d) of the leader's
last `L` bars × the follower's next bar. |T| ≥ 4.5 counts as a discovery. With ~264,000 pairs a
handful clear it by luck; the rolled maps (the leader shifted by whole sessions) measure how many.

**Windows: all available data** (user decision).
- Intraday runs 2025-06-18 → 2026-06-26; daily runs 2020-07-01 → 2026-06-12.
- Each is split into a discovery half and a validation half.
- A strategy built on this must validate on bars after 2026-06-26 (1m) / 2026-06-12 (1d).

**Pre-registered** pass bars are in `research.THRESH` and in the README's "Pre-registration"
section, written before the first run.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import hashlib
import importlib.util
import resource
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots
from sqlalchemy import text

from trading_simple import backtest as bt
from trading_simple import plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine
from trading_simple.features import load_features

settings = load_settings()
engine = make_engine(settings)
HERE = strategies.STRATEGIES_DIR / "lead_lag"
# research.py holds every computation; there is no strategy.py yet, so nothing here can trade.
_spec = importlib.util.spec_from_file_location("lead_lag_research", HERE / "research.py")
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)

SUBSET = None  # e.g. 40: a quick pass on 40 1m names and 200 daily names; None = everything
SEED = 0
BARS, SPEC, W, TH = rs.BARS, rs.SPEC, rs.WINDOWS, rs.THRESH
KIND = {"1m": "intraday", "5m": "intraday", "1d": "daily"}
COLOR = dict(zip(BARS, plots.SERIES, strict=False))
SEQ = [[0.0, "#dbe7f6"], [1.0, "#1b4f8f"]]  # one hue, light -> dark (magnitudes)
LINKS_TEXT = (HERE / "links.yaml").read_text()
LINKS = yaml.safe_load(LINKS_TEXT)
LINKS_SHA1 = hashlib.sha1(LINKS_TEXT.encode()).hexdigest()
T_START = time.time()
rng = np.random.default_rng(SEED)


def flat(a, part=None):
    # A cube as rows x columns (a session x bar series as rows), optionally cut to `part`.
    out = a.reshape(-1, a.shape[-1]) if a.ndim == 3 else a.reshape(-1)
    return out if part is None else out[part]


def below(fig, y=-0.14):
    # Legend under the plot: the default top-right legend sits on the subplot titles.
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


def trio(spacing=0.07):
    # One panel per bar size, side by side.
    return make_subplots(rows=1, cols=3, subplot_titles=list(BARS), horizontal_spacing=spacing)

## 1 · The data

* **1m pool.** Every name with ≥ 200 regular sessions of 1-minute bars in the window (the
  HighOctane pool).
  - It also takes the mega caps AAPL, MSFT, NVDA and TSLA. Their 1m bars start only in May/June
    2026, so they form a pre-registered leader *group* in §3 and are never screened.
  - Loaded **month by month**.
  - IBKR's volume-0 placeholder minutes are blanked before the 5-minute bars are derived, so every
    5m candle opens and closes on real prints.
* **Daily.** Every equity with a print on ≥ 95% of the window's days, plus SPY (the daily market)
  and GLD/SLV (leaders in §6's links test). Deep names (≥ 95% since 2015) feed §7's long-run figure.
* **Size, point-in-time.** Median dollar volume over the half-year *before* each window, and market
  cap (the SEC share count restated into split-adjusted units × close) at the window's start.

In [ ]:
with engine.connect() as conn:
    meta = pd.read_sql(
        text("SELECT symbol, security_type, sector, industry FROM securities"), conn
    ).set_index("symbol")
    sessions = pd.read_sql(
        text(
            "SELECT symbol, count(DISTINCT (ts AT TIME ZONE 'America/New_York')::date) AS n "
            "FROM prices_ohlcv WHERE bar = '1m' AND rth AND ts >= :a AND ts < :b GROUP BY symbol"
        ),
        conn,
        params={
            "a": pd.Timestamp(W["intraday"]["start"], tz="UTC"),
            "b": pd.Timestamp(W["intraday"]["end"], tz="UTC") + pd.Timedelta(days=1),
        },
    ).set_index("symbol")["n"]
ranked = sessions.sort_index().sort_values(ascending=False, kind="stable")
pool = ranked[ranked >= 200].index.tolist()
pool = sorted(pool[:SUBSET] if SUBSET else pool)
mega = [s for s in rs.MEGA if s in sessions.index and s not in pool]
cubes = rs.load_intraday(
    engine, pool + mega, W["intraday"]["start"], W["intraday"]["end"], minutes=(5,),
    skip_open=SPEC["1m"]["skip_open"], exchanges=settings.exchanges,
)
print(f"1m/5m: {len(pool)} names + mega {mega}, {cubes['1m']['x'].shape[0]} sessions "
      f"({time.time() - T_START:.0f} s)")

# Daily: every equity (SUBSET: the 200 deepest), the 1m names (for their size), SPY, GLD, SLV.
ETF = ["SPY", "GLD", "SLV"]
deepest = symbols_with_bars(
    engine, meta.index.tolist(), "1d", W["daily"]["long_run"], W["daily"]["end"]
)
equities = [s for s in deepest if meta.at[s, "security_type"] == "equity"]
equities = equities[:200] if SUBSET else equities
dpanel = load_panel(
    engine, sorted(set(equities) | set(pool) | set(mega)) + ETF, "1d", W["daily"]["long_run"],
    W["daily"]["end"], exchanges=settings.exchanges,
)
printed = (dpanel.volume > 0) & dpanel.close.notna()
cover = printed.loc[W["daily"]["start"]:].mean()
seen = dpanel.close.loc[slice(*W["daily"]["size"])].notna().sum()
daily_uni = [s for s in equities if cover[s] >= 0.95 and seen[s] >= 60]
deep = [s for s in daily_uni if printed[s].mean() >= 0.95]
cubes["1d"] = rs.cube(dpanel, symbols=daily_uni + ETF)

# Point-in-time size: dollar volume over the half-year before the window, market cap at its start.
dollar = dpanel.close * dpanel.volume
restate = strategies.load("turnover_momentum").signals.__globals__["_in_volume_units"]
shares = load_features(engine, dpanel.close.index, dpanel.symbols, ["fund_shares"])["fund_shares"]
cap = dpanel.close * restate(shares)


def table(names, kind, c):
    # Size, sector, industry, stale share and price of one bar size's names.
    a, b = W[kind]["size"]
    t = meta.reindex(names)[["sector", "industry"]].copy()
    t["dv"] = dollar.loc[a:b].median().reindex(names)
    t["cap"] = cap.loc[:b].ffill().iloc[-1].reindex(names)
    t["stale"] = c["stale"][: len(names)]
    t["price"] = c["price"][: len(names)]
    t["log_dv"], t["log_cap"] = np.log10(t["dv"]), np.log10(t["cap"])
    t["q_cap"] = pd.qcut(t["cap"].rank(method="first"), 5, labels=False)
    t["fast"] = (t["dv"].rank(pct=True) > 0.8) if kind == "intraday" else False
    return t


info = {b: table(pool if KIND[b] == "intraday" else daily_uni, KIND[b], cubes[b]) for b in BARS}
for b in ("1m", "5m"):
    cubes[b].pop("gap")  # intraday gaps are not used; frees memory
del dpanel, shares
print(f"1d: {len(daily_uni)} equities ({len(deep)} deep since 2015) + {ETF}; "
      f"{time.time() - T_START:.0f} s")

In [ ]:
overview = []
for bar in BARS:
    c, kind, t = cubes[bar], KIND[bar], info[bar]
    width, n = c["x"].shape[1], len(t)
    unit = width if kind == "intraday" else 1
    disc = rs.rows(c, W[kind]["start"], W[kind]["split"])
    val = rs.rows(c, W[kind]["split"])
    overview.append({
        "bar": bar,
        "names": n,
        "discovery": (disc.stop - disc.start) // unit,
        "validation": (val.stop - val.start) // unit,
        "unit": "sessions" if unit > 1 else "days",
        "bars per session": width if unit > 1 else 1,
        "median stale share": np.nanmedian(t["stale"]),
        "median price ($)": t["price"].median(),
        "median $vol ($M/day)": t["dv"].median() / 1e6,
        "names with a market cap": int(t["cap"].notna().sum()),
    })
display(pd.DataFrame(overview).set_index("bar").round(3))

t = info["1m"]
fig = go.Figure(go.Scatter(
    x=t["log_dv"], y=t["stale"], mode="markers", text=t.index,
    marker={"size": 8, "color": np.log10(t["price"]), "colorscale": SEQ,
            "colorbar": {"title": "log10 price"}, "line": {"color": "white", "width": 1}},
    hovertemplate="%{text}<br>log10 $vol %{x:.2f}<br>no print %{y:.1%}<extra></extra>",
))
fig.update_xaxes(title="log10 median dollar volume per day, 2025-H1 (point-in-time)")
fig.update_yaxes(title="share of minutes without a print", tickformat=".0%")
plots._base(fig, title="Fig 1 · at 1m, 'big' and 'prints every minute' are nearly one variable",
            height=460)
fig.show()
print(f"Spearman(no-print share, log $vol) = {rs.spearman(t['stale'], t['log_dv']):+.2f}, "
      f"(no-print share, log price) = {rs.spearman(t['stale'], np.log(t['price'])):+.2f}")

## 2 · Units, costs and power

**Units.** Returns are put in **causal volatility units**: each name's trailing RMS (5 sessions;
60 days at 1d) × a pooled time-of-day profile fitted on discovery. The open moves several times
more than midday (Fig 2) and would otherwise carry every correlation.

**Power, before any map.** Two numbers per bar size:
- the correlation a trade would need to pay its round trip (`bt.leg_costs` at the universe's
  median price, $10k orders);
- the smallest correlation the map can detect on the discovery half.

If the second is larger, the pair map cannot see a tradable pair at that bar size, and its pair
checks read *n/a*. §4's canary confirms this empirically.

In [ ]:
D, rt, power = {}, {}, []
for bar in BARS:
    c, sp, kind, t = cubes[bar], SPEC[bar], KIND[bar], info[bar]
    n, width = len(t), c["x"].shape[1]
    disc = rs.rows(c, W[kind]["start"], W[kind]["split"])
    fit = slice(disc.start // width, disc.stop // width) if kind == "intraday" else None
    xs = rs.scale(c["x"], days=sp["days"], fit=fit)  # every column (mega / ETF leaders too)
    ys = rs.scale(c["y"][..., :n], days=sp["days"], fit=fit)
    if kind == "intraday":  # the fast market: equal-weight of the most-traded fifth
        m = np.nanmean(xs[..., :n][..., t["fast"].to_numpy()], axis=-1)
    else:
        m = xs[..., c["symbols"].index("SPY")]
    m = np.nan_to_num(m)
    D[bar] = d = {
        "c": c, "n": n, "width": width, "disc": disc, "val": rs.rows(c, W[kind]["split"]),
        "full": rs.rows(c, W[kind]["start"]),
        "cl": rs.clusters(c, "session" if kind == "intraday" else "week"),
        "xs": xs, "ys": ys, "m": m,
        "shared": [m, rs.lagged(m, 1, sp["L"]), rs.lagged(m, sp["L"] + 1, sp["K"])],
        "own": rs.lagged(xs[..., :n], 1, sp["L"]),
    }
    fees, slip = bt.leg_costs(settings.costs, price=float(np.nanmedian(c["price"][:n])))
    rt[bar] = 2 * (fees + slip)
    f = flat(rs.forward(c["x"][..., :n], c["y"][..., :n], sp["hold"]), disc)
    sigma = float(np.nanmedian(np.nanstd(f, axis=0)))
    obs = float(np.median(np.isfinite(flat(ys, disc)).sum(axis=0)))
    rho_h = rt[bar] / (rs.Z_TRADE * sigma)
    d["rho_1"] = rho_h / np.sqrt(sp["hold"])
    d["rho_det"] = (rs.T_CUT + 0.84) / np.sqrt(obs)
    d["informative"] = bool(d["rho_1"] >= d["rho_det"])
    power.append({
        "bar": bar, "round trip (bp)": 1e4 * rt[bar], "hold (bars)": sp["hold"],
        "σ of the hold (bp)": 1e4 * sigma, "break-even ρ, hold": rho_h,
        "break-even ρ per bar": d["rho_1"], "observations per name": obs,
        "detectable ρ (|T| 4.5, 80%)": d["rho_det"], "pair map informative": d["informative"],
    })
    del f
display(pd.DataFrame(power).set_index("bar").round(4))

fig = go.Figure()
for bar in ("1m", "5m"):
    c, n = D[bar]["c"], D[bar]["n"]
    rms = 1e4 * np.sqrt(np.nanmean(np.square(c["x"][..., :n]), axis=(0, 2)))
    fig.add_trace(go.Scatter(
        x=np.arange(len(rms)) * int(bar[:-1]), y=rms, mode="lines", name=f"{bar} bars",
        line={"color": COLOR[bar], "width": 2},
        hovertemplate="minute %{x}<br>%{y:.1f} bp<extra></extra>",
    ))
fig.update_xaxes(title="minutes since the open")
fig.update_yaxes(title="RMS last-trade return (bp)", type="log")
plots._base(fig, title="Fig 2 · the open moves several times more than midday — scaled out first",
            height=380)
fig.show()

## 3 · Does the market lead? (C1)

The simplest lead-lag: the **fast market** moves, and everyone else follows a bar or two later. The
fast market is the equal-weight of the most-traded fifth of names (point-in-time dollar volume); at
1d it is SPY.

Each name's *tradable* next bar is regressed (Granger form) on:
- the fast market now;
- its last `L` bars;
- its lags `L+1..K`;
- the name's own last `L` bars.

The **lag t** is the t of the fast market's last `L` bars. Pooled over all non-fast names it is C1;
per name it is §5's response delay.

The mega caps (their May–June 2026 sessions only) are tested as one extra leader group on top of the
fast market. Fig 3 draws the whole lag profile per size fifth.

In [ ]:
def pooled(d, y, own, part, shared=None):
    # Granger fit of equal-weight columns on the rows `part`.
    shared = d["shared"] if shared is None else shared
    return rs.granger(flat(y, part), [flat(a, part) for a in shared], own=flat(own, part),
                      cl=d["cl"][part])


market = []
for bar in BARS:
    d, sp, t = D[bar], SPEC[bar], info[bar]
    full = d["full"]
    g = rs.granger(flat(d["ys"], full), [flat(a, full) for a in d["shared"]],
                   own=flat(d["own"], full), cl=d["cl"][full])
    d["u_full"], d["lag_t"], d["coef_full"] = g["u"], g["t"][:, 2], g["coef"]
    rest = np.where(t["fast"], None, "rest")
    d["P"], _ = rs.portfolios(d["ys"], rest)
    d["Pown"], _ = rs.portfolios(d["own"], rest)
    gp = pooled(d, d["P"], d["Pown"], full)
    row = {"bar": bar, "C1: pooled lag t": gp["t"][0, 2], "pooled lag coef": gp["coef"][0, 2],
           "names with lag t ≥ 2": np.mean(d["lag_t"] >= 2),
           "names with lag t ≤ −2": np.mean(d["lag_t"] <= -2)}
    # Does the fast market continue its own move? (its tradable next bar on its own last L bars)
    if KIND[bar] == "intraday":
        pf, _ = rs.portfolios(d["ys"], np.where(t["fast"], "fast", None))
    else:
        spy = d["c"]["symbols"].index("SPY")
        pf = rs.scale(d["c"]["y"][..., [spy]], days=sp["days"])
    row["fast market's own continuation t"] = rs.granger(
        flat(pf, full), [flat(a, full) for a in d["shared"]], cl=d["cl"][full])["t"][0, 2]
    if KIND[bar] == "intraday" and mega:  # the mega caps as ONE leader group, on their sessions
        mc = d["xs"][..., d["n"]:]
        mg = np.where(np.isfinite(mc).sum(-1) >= min(3, len(mega)), np.nanmean(mc, axis=-1), np.nan)
        live = np.isfinite(mg).mean(axis=1) > 0.5
        y = np.where(live[:, None, None], d["P"], np.nan)
        extra = [np.nan_to_num(mg), rs.lagged(mg, 1, sp["L"])]
        gm = pooled(d, y, d["Pown"], full, shared=d["shared"] + extra)
        row.update({"MEGA sessions": int(live.sum()), "MEGA → rest lag t": gm["t"][0, 5]})
    market.append(row)
market = pd.DataFrame(market).set_index("bar")
display(market.round(3))

fig = trio()
for i, bar in enumerate(BARS, start=1):
    d, sp, t = D[bar], SPEC[bar], info[bar]
    lags = [d["m"]] + [rs.lagged(d["m"], k, k) for k in range(1, sp["K"] + 1)]
    pq, groups = rs.portfolios(d["ys"], t["q_cap"].to_numpy())
    pq_own, _ = rs.portfolios(d["own"], t["q_cap"].to_numpy())
    g = pooled(d, pq, pq_own, d["full"], shared=lags)
    for gi, grp in enumerate(groups):
        tag = {0: " (small)", 4: " (big)"}.get(int(grp), "")
        fig.add_trace(go.Scatter(
            x=list(range(1, sp["K"] + 1)), y=g["coef"][gi, 2:2 + sp["K"]], mode="lines+markers",
            name=f"fifth {int(grp) + 1}{tag}", legendgroup=str(grp), showlegend=i == 1,
            line={"color": plots.SERIES[gi], "width": 2}, marker={"size": 8},
            hovertemplate="lag %{x}<br>response %{y:.4f}<extra></extra>",
        ), row=1, col=i)
    fig.add_hline(y=0, line={"color": plots._MUTED, "width": 1}, row=1, col=i)
    fig.update_xaxes(title="lag k (bars)", row=1, col=i)
fig.update_yaxes(title="response to the fast market k bars ago", row=1, col=1)
plots._base(fig, title="Fig 3 · each cap fifth's tradable next bar vs the fast market's past moves",
            height=440)
below(fig).show()

## 4 · The pair map (discovery half)

Every leader × follower at once, on the **discovery** half, three ways:

* **naive**: the leader's last `L` close-to-close returns × the follower's next close-to-close
  return. This is what a plain lagged-correlation screen shows; stale prints and the market both
  live in it.
* **tradable**: the same, but the follower's target is its next bar's own first-to-last print
  (entry at the first print after the signal). The stale catch-up is gone; the market is still in.
* **beyond the market**: the tradable target's Granger innovation × the leader net of its same-bar
  market move. **This is the map every check reads.**

The null is the same map with the leader rolled by whole sessions (days at 1d). That destroys the
alignment and keeps everything else.
- **V1** checks that the rolls look like a t: std ≈ 1 and thin tails.
- **C2** asks for more discoveries than any roll, a small empirical false-discovery rate, and no
  single follower owning them.
- **V2** plants 20 leads at the break-even correlation and asks whether the map finds them.

In [ ]:
GRID = [3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0, 8.0]
CUT = GRID.index(rs.T_CUT)


def off(T):
    # Off-diagonal finite entries of a map.
    keep = np.isfinite(T)
    if T.shape[0] == T.shape[1]:
        np.fill_diagonal(keep, False)
    return T[keep]


pairmap = []
for bar in BARS:
    d, sp = D[bar], SPEC[bar]
    n, disc = d["n"], d["disc"]
    cl = d["cl"][disc]
    xs_u, m = d["xs"][..., :n], d["m"]
    xd, md = flat(xs_u, disc), flat(m, disc)
    ok = np.isfinite(xd)
    beta = ((np.where(ok, xd, 0) * md[:, None]).sum(0) / (ok * md[:, None] ** 2).sum(0))
    beta = beta.astype(np.float32)
    d["e"] = xs_u - beta * m[..., None]  # the leader net of its same-bar market move
    d["E"] = rs.lagged(d["e"], 1, sp["L"])
    Ed = flat(d["E"], disc)
    gd = rs.granger(flat(d["ys"], disc), [flat(a, disc) for a in d["shared"]],
                    own=flat(d["own"], disc), cl=cl)
    d["u_disc"], d["lag_t_disc"] = gd["u"], gd["t"][:, 2]
    d["Td"] = Td = rs.pair_t(Ed, gd["u"], cl)
    np.fill_diagonal(Td, np.nan)
    naive = rs.pair_t(flat(d["own"], disc), flat(xs_u, disc), cl)
    tradable = rs.pair_t(flat(d["own"], disc), flat(d["ys"], disc), cl)
    d["counts"] = {name: [int((np.abs(off(T)) >= g).sum()) for g in GRID]
                   for name, T in (("naive", naive), ("tradable", tradable),
                                   ("beyond market", Td))}
    del naive, tradable
    unit = d["width"] if KIND[bar] == "intraday" else 1
    span = (disc.stop - disc.start) // unit
    d["rolls"] = []
    for _ in range(sp["rolls"]):
        k = int(rng.integers(sp["min_roll"], span - sp["min_roll"]))
        Tr = rs.pair_t(Ed, gd["u"], cl, roll=k * unit)
        np.fill_diagonal(Tr, np.nan)
        a = off(Tr)
        top = np.argsort(np.nan_to_num(np.abs(Tr), nan=-1.0), axis=None)[-100:]
        d["rolls"].append({"std": float(a.std()), "tail": float((np.abs(a) >= rs.T_CUT).mean()),
                           "counts": [int((np.abs(a) >= g).sum()) for g in GRID],
                           "top": top, "sign": np.sign(Tr.ravel()[top])})
    rc = np.array([r["counts"] for r in d["rolls"]])
    real = d["counts"]["beyond market"][CUT]
    found = np.abs(np.nan_to_num(Td)) >= rs.T_CUT
    # The canary: 20 leads planted at the break-even correlation must be found.
    lead = rng.choice(n, 20)
    follow = (lead + rng.integers(1, n, 20)) % n
    EI, UJ = Ed[:, lead], gd["u"][:, follow]
    amp = d["rho_1"] * np.nanstd(UJ, axis=0) / np.nanstd(EI, axis=0)
    canary = np.diag(rs.pair_t(EI, UJ + amp * EI, cl))
    d["canary"] = int((canary >= rs.T_CUT).sum())
    stds = [r["std"] for r in d["rolls"]]
    pairmap.append({
        "bar": bar, "pairs": int(off(Td).size),
        "naive ≥ 4.5": d["counts"]["naive"][CUT],
        "tradable ≥ 4.5": d["counts"]["tradable"][CUT],
        "beyond market ≥ 4.5": real,
        "roll mean": rc[:, CUT].mean(), "roll max": int(rc[:, CUT].max()),
        "FDR": rc[:, CUT].mean() / real if real else np.nan,
        "busiest follower share": found.sum(0).max() / real if real else np.nan,
        "roll T std min": min(stds), "roll T std max": max(stds),
        "roll tail share max": max(r["tail"] for r in d["rolls"]),
        "canary /20": d["canary"], "informative (§2)": d["informative"],
    })
    d["roll_counts"] = rc
pairmap = pd.DataFrame(pairmap).set_index("bar")
display(pairmap.T)

fig = trio()
for i, bar in enumerate(BARS, start=1):
    rc = D[bar]["roll_counts"]
    fig.add_trace(go.Scatter(
        x=GRID + GRID[::-1], y=list(rc.max(0) + 1) + list((rc.min(0) + 1)[::-1]), fill="toself",
        mode="lines", fillcolor="rgba(137,135,129,0.25)", line={"width": 0},
        name="rolled nulls (min–max)",
        legendgroup="null", showlegend=i == 1, hoverinfo="skip",
    ), row=1, col=i)
    for j, (name, cnt) in enumerate(D[bar]["counts"].items()):
        fig.add_trace(go.Scatter(
            x=GRID, y=np.array(cnt) + 1, mode="lines+markers", name=name, legendgroup=name,
            showlegend=i == 1, line={"color": plots.SERIES[j], "width": 2}, marker={"size": 8},
            hovertemplate="|T| ≥ %{x}<br>%{y} pairs (+1)<extra></extra>",
        ), row=1, col=i)
    fig.add_vline(x=rs.T_CUT, line={"color": plots._MUTED, "dash": "dot"}, row=1, col=i)
    fig.update_xaxes(title="|T| at least", row=1, col=i)
    fig.update_yaxes(type="log", row=1, col=i)
fig.update_yaxes(title="pairs + 1 (log)", row=1, col=1)
plots._base(fig, title="Fig 4a · discoveries by threshold — naive → tradable → beyond the market, "
                       "against rolled nulls", height=440)
below(fig).show()

fig = trio(spacing=0.13)
blocks = {}
for bar in BARS:
    sec = info[bar]["sector"].fillna("?").to_numpy()
    groups = sorted(set(sec))
    H = np.stack([sec == g for g in groups], axis=1).astype(float)
    Td = D[bar]["Td"]
    blocks[bar] = (groups, (H.T @ np.nan_to_num(Td) @ H) / np.maximum(H.T @ np.isfinite(Td) @ H, 1))
zmax = max(np.nanmax(np.abs(z)) for _, z in blocks.values())
for i, bar in enumerate(BARS, start=1):
    groups, z = blocks[bar]
    short = [g[:16] for g in groups]
    fig.add_trace(go.Heatmap(
        z=z, x=short, y=short, colorscale=plots.DIVERGING, zmid=0, zmin=-zmax, zmax=zmax,
        showscale=i == 3, colorbar={"title": "mean T"},
        hovertemplate="leader %{y}<br>follower %{x}<br>mean T %{z:.2f}<extra></extra>",
    ), row=1, col=i)
plots._base(fig, title="Fig 4b · mean T by sector — leader (rows) → follower (columns), discovery, "
                       "beyond the market", height=560)
fig.show()

## 5 · Size and staleness (C4)

**Who lags the market?** Each name's **lag t** (§3, full window) is set against its point-in-time
**market cap**. Two things masquerade as size, so both are controlled for:
- how often the name **prints** (stale share);
- its **price**, the odd-lot channel (IBKR counts round lots only).

Dollar volume is the rival explanation (Chordia–Swaminathan).

**The Lo–MacKinlay test.** Build equal-weight portfolios of the five cap fifths. Set each fifth's
tradable next bar, net of its own lags, against every fifth's last `L` bars. Does **big → small**
beat **small → big**?

In [ ]:
def rank_t(y, *xs):
    # OLS t's of y on the regressors, everything as ranks (an intercept added).
    frame = pd.concat([pd.Series(np.asarray(v, float)) for v in (y, *xs)], axis=1).dropna().rank()
    if len(frame) < 10:
        return np.full(len(xs), np.nan)
    Y = frame.iloc[:, 0].to_numpy()
    X = np.column_stack([np.ones(len(frame)), frame.iloc[:, 1:].to_numpy()])
    b = np.linalg.lstsq(X, Y, rcond=None)[0]
    resid = Y - X @ b
    s2 = resid @ resid / (len(Y) - X.shape[1])
    return b[1:] / np.sqrt(np.diag(s2 * np.linalg.pinv(X.T @ X))[1:])


size = []
for bar in BARS:
    d, sp, t = D[bar], SPEC[bar], info[bar]
    lag_t = d["lag_t"]
    t_cap = rank_t(lag_t, t["log_cap"], t["stale"], np.log(t["price"]))
    t_dv = rank_t(lag_t, t["log_dv"], t["stale"], np.log(t["price"]))
    q = t["q_cap"].to_numpy()
    px, _ = rs.portfolios(d["xs"][..., :d["n"]], q)
    py, _ = rs.portfolios(d["ys"], q)
    lx = rs.lagged(px, 1, sp["L"])
    full = d["full"]
    u5 = rs.granger(flat(py, full), [], own=flat(lx, full), cl=d["cl"][full])["u"]
    d["T5"] = T5 = rs.pair_t(flat(lx, full), u5, d["cl"][full])
    # post-hoc (not pre-registered): the SHARE of a name's market response that arrives late.
    # A t mixes delay with precision (a name the market explains well has a larger t).
    now, late = d["coef_full"][:, 1], d["coef_full"][:, 2]
    share = np.where(now + late > 0, late / (now + late), np.nan)
    size.append({
        "bar": bar, "Spearman(lag t, log cap)": rs.spearman(lag_t, t["log_cap"]),
        "cap t | stale, price": t_cap[0], "stale t | cap, price": t_cap[1],
        "Spearman(lag t, log $vol)": rs.spearman(lag_t, t["log_dv"]),
        "$vol t | stale, price": t_dv[0], "Spearman(lag t, stale)": rs.spearman(lag_t, t["stale"]),
        "T big → small": T5[4, 0], "T small → big": T5[0, 4],
        "post-hoc: Spearman(late share, log cap)": rs.spearman(share, t["log_cap"]),
        "post-hoc: Spearman(late share, no-print)": rs.spearman(share, t["stale"]),
        "post-hoc: late share t | stale, price (cap)":
            rank_t(share, t["log_cap"], t["stale"], np.log(t["price"]))[0],
    })
size = pd.DataFrame(size).set_index("bar")
display(size.round(3))

fig = trio()
for i, bar in enumerate(BARS, start=1):
    t = info[bar]
    fig.add_trace(go.Scatter(
        x=t["log_cap"], y=D[bar]["lag_t"], mode="markers", text=t.index, showlegend=False,
        marker={"size": 8, "color": t["stale"], "colorscale": SEQ, "cmin": 0, "cmax": 0.5,
                "showscale": i == 3, "colorbar": {"title": "no print", "tickformat": ".0%"},
                "line": {"color": "white", "width": 1}},
        hovertemplate="%{text}<br>log10 cap %{x:.2f}<br>lag t %{y:.1f}<extra></extra>",
    ), row=1, col=i)
    fig.add_hline(y=0, line={"color": plots._MUTED, "width": 1}, row=1, col=i)
    fig.update_xaxes(title="log10 market cap", row=1, col=i)
fig.update_yaxes(title="lag t (response to the fast market's last L bars)", row=1, col=1)
plots._base(fig, title="Fig 5a · who lags the market, by size — coloured by how often a name "
                       "fails to print", height=440)
fig.show()

fig = trio(spacing=0.1)
labels = ["1 small", "2", "3", "4", "5 big"]
zmax = max(np.nanmax(np.abs(D[b]["T5"])) for b in BARS)
for i, bar in enumerate(BARS, start=1):
    T5 = D[bar]["T5"]
    fig.add_trace(go.Heatmap(
        z=T5, x=labels, y=labels, colorscale=plots.DIVERGING, zmid=0, zmin=-zmax, zmax=zmax,
        texttemplate="%{z:.1f}", showscale=i == 3, colorbar={"title": "T"},
        hovertemplate="leader %{y}<br>follower %{x}<br>T %{z:.2f}<extra></extra>",
    ), row=1, col=i)
    fig.update_xaxes(title="follower (cap fifth)", row=1, col=i)
fig.update_yaxes(title="leader (cap fifth)", row=1, col=1)
plots._base(fig, title="Fig 5b · Lo–MacKinlay — each cap fifth's last L bars → every fifth's next "
                       "bar (own lags out)", height=420)
fig.show()

## 6 · Links (C5)

Three economic catalysts, each tested on the full window's map *beyond the market*. A
pre-registered hypothesis needs no discovery half.

* **a · industries.** Is a leader's pull on followers in its **own** Nasdaq industry stronger than
  on the rest? (1,000 permutations of the industry labels.)
* **b · Hou (2007).** Inside each industry with ≥ 6 names, does the **big** half lead the **small**
  half more than the reverse?
* **c · supply chains and metals.** The customer → supplier and GLD/SLV → miner edges in
  `links.yaml`, written before any map.
  - For each edge: the leader's **percentile** among all leaders of that follower (0.5 = no better
    than a random leader).
  - It is read on the fast map (lag `L`) and on a **slow** daily map: lags 2..22 against
    close-to-close innovations, monthly clusters. Customer → supplier drift is a monthly effect
    (Cohen–Frazzini).
  - The reverse direction (supplier → customer) is the placebo.
  - Tested at 1d; the intraday bars report coverage only.

In [ ]:
def industry_test(T, codes, perms=1000):
    # Same- minus cross-industry mean T, and its percentile among label permutations.
    ok = np.isfinite(T)
    Tz = np.where(ok, T, 0.0)
    has = np.flatnonzero(codes >= 0)
    tot_s, tot_n = Tz[np.ix_(has, has)].sum(), ok[np.ix_(has, has)].sum()

    def stat(lab):
        s = k = 0.0
        for g in np.unique(lab[has]):
            idx = has[lab[has] == g]
            s += Tz[np.ix_(idx, idx)].sum()
            k += ok[np.ix_(idx, idx)].sum()
        return s / k - (tot_s - s) / (tot_n - k)

    real = stat(codes)
    null = []
    for _ in range(perms):
        p = codes.copy()
        p[has] = rng.permutation(codes[has])
        null.append(stat(p))
    return real, float((np.array(null) < real).mean())


def percentile(T, i, j):
    # The leader's percentile among the other leaders of follower j.
    col = T[:, j]
    keep = np.isfinite(col)
    keep[i] = False
    return float((col[keep] < T[i, j]).mean()) if np.isfinite(T[i, j]) else np.nan


edges = [(e["leader"], e["follower"], kind) for kind, es in LINKS.items() for e in es]
links_rows, link_tables = [], {}
for bar in BARS:
    d, sp, t = D[bar], SPEC[bar], info[bar]
    n, full = d["n"], d["full"]
    cl = d["cl"][full]
    Ef = flat(d["E"], full)
    d["Tf"] = Tf = rs.pair_t(Ef, d["u_full"], cl)
    np.fill_diagonal(Tf, np.nan)
    real, pct = industry_test(Tf, pd.factorize(t["industry"])[0])
    row = {"bar": bar, "industry: same − cross mean T": real, "industry: permutation pct": pct}
    # b · Hou: the big vs the small half inside each industry with >= 6 names
    lab = pd.Series(None, index=t.index, dtype=object)
    for ind, g in t.dropna(subset=["cap", "industry"]).groupby("industry"):
        if len(g) >= 6:
            lab[g.index] = np.where(g["cap"] > g["cap"].median(), f"{ind}|big", f"{ind}|small")
    diffs = []
    if lab.notna().any():
        pe, groups = rs.portfolios(d["e"], lab.to_numpy())
        pu, _ = rs.portfolios(d["u_full"], lab.to_numpy())
        Th = rs.pair_t(flat(rs.lagged(pe, 1, sp["L"]), full), pu, cl)
        for ind in sorted({g.split("|")[0] for g in groups}):
            if f"{ind}|big" in groups and f"{ind}|small" in groups:
                b, s = groups.index(f"{ind}|big"), groups.index(f"{ind}|small")
                diffs.append(Th[b, s] - Th[s, b])
    diffs = np.array(diffs)
    row.update({
        "Hou industries": len(diffs),
        "Hou mean (big→small − small→big)": diffs.mean() if len(diffs) else np.nan,
        "Hou t": diffs.mean() / diffs.std(ddof=1) * np.sqrt(len(diffs))
        if len(diffs) > 2 else np.nan,
    })
    # c · links: the fast map (+ GLD/SLV as leaders at 1d) and, at 1d, the slow map
    names, leaders = list(t.index), list(t.index)
    maps = {"fast": Tf}
    if KIND[bar] == "daily":
        sym = d["c"]["symbols"]
        xe = d["xs"][..., [sym.index("GLD"), sym.index("SLV")]]
        xd, md = flat(xe, d["disc"]), flat(d["m"], d["disc"])
        ok = np.isfinite(xd)
        be = ((np.where(ok, xd, 0) * md[:, None]).sum(0) / (ok * md[:, None] ** 2).sum(0))
        be = be.astype(np.float32)
        ee = xe - be * d["m"][..., None]
        leaders = names + ["GLD", "SLV"]
        maps["fast"] = np.vstack([Tf, rs.pair_t(flat(rs.lagged(ee, 1, sp["L"]), full),
                                               d["u_full"], cl)])
        lo, hi = rs.SLOW
        cm = rs.clusters(d["c"], "month")[full]
        u_slow = rs.granger(
            flat(d["xs"][..., :n], full),
            [flat(d["m"], full), flat(rs.lagged(d["m"], 1, hi), full)],
            own=flat(rs.lagged(d["xs"][..., :n], 1, hi), full), cl=cm,
        )["u"]
        slow = rs.pair_t(np.hstack([flat(rs.lagged(d["e"], lo, hi), full),
                                    flat(rs.lagged(ee, lo, hi), full)]), u_slow, cm)
        slow[:n][np.eye(n, dtype=bool)] = np.nan
        maps["slow"] = slow
        del u_slow
    lead_ix = {s: i for i, s in enumerate(leaders)}
    fol_ix = {s: j for j, s in enumerate(names)}
    covered = [(a, b, k) for a, b, k in edges if a in lead_ix and b in fol_ix and a != b]
    rows_l = []
    for a, b, k in covered:
        i, j = lead_ix[a], fol_ix[b]
        r = {"leader": a, "follower": b, "kind": k}
        for name, T in maps.items():
            r[f"T {name}"], r[f"pct {name}"] = T[i, j], percentile(T, i, j)
            if b in lead_ix and a in fol_ix:  # the placebo: supplier -> customer
                r[f"pct {name} (reverse)"] = percentile(T, lead_ix[b], fol_ix[a])
        rows_l.append(r)
    lt = link_tables[bar] = pd.DataFrame(rows_l)
    row["links covered"] = len(covered)
    for name, T in maps.items():
        if len(lt) >= 5:
            js = np.array([fol_ix[b] for _, b, _ in covered])
            m_j = np.maximum(np.isfinite(T[:, js]).sum(0) - 1, 1)
            draws = (rng.integers(0, m_j, size=(10_000, len(js))) / m_j).mean(axis=1)
            obs = lt[f"pct {name}"].mean()
            row.update({f"links: mean pct {name}": obs,
                        f"links: p {name}": float((draws >= obs).mean()),
                        f"links: reverse pct {name}": lt.get(f"pct {name} (reverse)",
                                                             pd.Series(dtype=float)).mean()})
    links_rows.append(row)
links = pd.DataFrame(links_rows).set_index("bar")
display(links.T.round(4))
print("Links at 1d (pct = the leader's percentile among all leaders of that follower):")
display(link_tables["1d"].groupby("kind").agg(
    links=("leader", "size"), **{c: (c, "mean") for c in link_tables["1d"].columns
                                 if c.startswith("pct")}).round(3))
display(link_tables["1d"].round(3))

lt = link_tables["1d"]
fig = go.Figure()
for j, col in enumerate([c for c in ("pct fast", "pct slow", "pct fast (reverse)",
                                     "pct slow (reverse)") if c in lt]):
    fig.add_trace(go.Box(
        y=lt[col], name=col.replace("pct ", ""), boxpoints="all", jitter=0.4, pointpos=0,
        marker={"color": plots.SERIES[j], "size": 8}, line={"color": plots.SERIES[j]},
        text=lt["leader"] + " → " + lt["follower"],
        hovertemplate="%{text}<br>percentile %{y:.2f}<extra></extra>",
    ))
fig.add_hline(y=0.5, line={"color": plots._MUTED, "dash": "dot"})
fig.update_yaxes(title="leader's percentile among all leaders of that follower", range=[0, 1])
plots._base(fig, title="Fig 6 · pre-registered links at 1d — 0.5 is a random leader", height=440)
fig.show()

## 7 · Persistence (C3) — and the screen

Anything *selected* on the discovery half is judged on the validation half. The validation half is
fitted afresh with its own Granger controls; only the leader's market beta is kept from discovery.

* **Market.** Do the same names lag? This reads the Spearman of each name's lag t, discovery vs
  validation, and the pooled lag t on validation alone.
* **Pairs.** The discovery map's **top 100** pairs: their validation T, signed by the discovery
  sign. The null is the same statistic for the top 100 of each *rolled* discovery map, i.e. pairs
  selected on luck.

**The screen table** lists the top 100 with everything a strategy designer needs: industries, size
ranks, stale shares, the best lag, discovery and validation T, and whether the pair is a
pre-registered link. Fig 7b follows the daily market lead and big → small through 2016–2026 on the
deep names.

In [ ]:
persist, screens = [], {}
link_set = {(a, b) for a, b, _ in edges}
for bar in BARS:
    d, sp, t = D[bar], SPEC[bar], info[bar]
    val, cl = d["val"], d["cl"][d["val"]]
    gv = rs.granger(flat(d["ys"], val), [flat(a, val) for a in d["shared"]],
                    own=flat(d["own"], val), cl=cl)
    d["lag_t_val"] = gv["t"][:, 2]
    d["Tv"] = Tv = rs.pair_t(flat(d["E"], val), gv["u"], cl)
    np.fill_diagonal(Tv, np.nan)
    Td = d["Td"]
    both = np.isfinite(Td) & np.isfinite(Tv)
    top = np.argsort(np.where(both, np.abs(Td), -1.0), axis=None)[-100:][::-1]
    signed = np.sign(Td.ravel()[top]) * Tv.ravel()[top]
    null = []
    for r in d["rolls"]:
        s = r["sign"] * Tv.ravel()[r["top"]]
        s = s[np.isfinite(s)]
        null.append((s.mean(), (s > 0).mean()))
    null = np.array(null)
    persist.append({
        "bar": bar, "lag t Spearman disc → val": rs.spearman(d["lag_t_disc"], d["lag_t_val"]),
        "pooled lag t (validation)": pooled(d, d["P"], d["Pown"], val)["t"][0, 2],
        "top-100 mean signed T (val)": signed.mean(), "top-100 same sign": (signed > 0).mean(),
        "null 95% mean T": np.quantile(null[:, 0], 0.95),
        "null 95% same sign": np.quantile(null[:, 1], 0.95),
    })
    i, j = np.unravel_index(top, Td.shape)
    ed, ud, cd = d["disc"], d["u_disc"], d["cl"][d["disc"]]
    per_lag = np.stack([np.diag(rs.pair_t(flat(rs.lagged(d["e"], k, k), ed)[:, i], ud[:, j], cd))
                        for k in range(1, sp["L"] + 1)])
    names, ind = t.index.to_numpy(), t["industry"].to_numpy()
    cap_pct, stale = t["cap"].rank(pct=True).to_numpy(), t["stale"].to_numpy()
    screens[bar] = pd.DataFrame({
        "leader": names[i], "follower": names[j], "leader industry": ind[i],
        "follower industry": ind[j], "same industry": ind[i] == ind[j],
        "leader cap pct": cap_pct[i], "follower cap pct": cap_pct[j],
        "leader no-print": stale[i], "follower no-print": stale[j],
        "best lag": np.argmax(np.abs(per_lag), axis=0) + 1,
        "T discovery": Td[i, j], "T validation": Tv[i, j],
        "link": [(a, b) in link_set for a, b in zip(names[i], names[j], strict=True)],
    })
persist = pd.DataFrame(persist).set_index("bar")
display(persist.round(3))
for bar in ("1m", "5m"):
    print(f"The screen at {bar}: the discovery map's 20 strongest pairs "
          f"(all 100 in screens['{bar}'])")
    display(screens[bar].head(20).round(3))

fig = trio()
for k, bar in enumerate(BARS, start=1):
    Td, Tv = D[bar]["Td"], D[bar]["Tv"]
    ok = np.flatnonzero((np.isfinite(Td) & np.isfinite(Tv)).ravel())
    pick = rng.choice(ok, min(3000, len(ok)), replace=False)
    fig.add_trace(go.Scatter(
        x=Td.ravel()[pick], y=Tv.ravel()[pick], mode="markers", name="random pairs",
        legendgroup="r", showlegend=k == 1, hoverinfo="skip",
        marker={"size": 5, "color": plots._MUTED, "opacity": 0.35},
    ), row=1, col=k)
    s = screens[bar]
    fig.add_trace(go.Scatter(
        x=s["T discovery"], y=s["T validation"], mode="markers", name="discovery top 100",
        legendgroup="t", showlegend=k == 1, text=s["leader"] + " → " + s["follower"],
        marker={"size": 8, "color": plots.SERIES[0], "line": {"color": "white", "width": 1}},
        hovertemplate="%{text}<br>T discovery %{x:.1f}<br>T validation %{y:.1f}<extra></extra>",
    ), row=1, col=k)
    fig.add_hline(y=0, line={"color": plots._MUTED, "width": 1}, row=1, col=k)
    fig.update_xaxes(title="T, discovery", row=1, col=k)
fig.update_yaxes(title="T, validation", row=1, col=1)
plots._base(fig, title="Fig 7a · does a pair's discovery T carry into validation? "
                       "(beyond the market)", height=440)
below(fig).show()

# Fig 7b: the daily market lead and big -> small, year by year, on the deep names.
d, sp, t = D["1d"], SPEC["1d"], info["1d"]
names = list(t.index)
is_deep = np.isin(names, deep)
pd_, _ = rs.portfolios(d["ys"], np.where(is_deep, "deep", None))
pd_own, _ = rs.portfolios(d["own"], np.where(is_deep, "deep", None))
years = []
for y in range(2016, int(W["daily"]["end"][:4]) + 1):
    r = rs.rows(d["c"], f"{y}-01-01", f"{y + 1}-01-01")
    g = pooled(d, pd_, pd_own, r)
    dv_prev = dollar.loc[f"{y - 1}-01-01":f"{y - 1}-12-31"].median().reindex(deep)
    q = pd.Series(np.nan, index=names)
    q[deep] = pd.qcut(dv_prev.rank(method="first"), 5, labels=False).to_numpy()
    px, _ = rs.portfolios(d["xs"][..., :d["n"]], q.to_numpy())
    py, _ = rs.portfolios(d["ys"], q.to_numpy())
    lx = rs.lagged(px, 1, sp["L"])
    u5 = rs.granger(flat(py, r), [], own=flat(lx, r), cl=d["cl"][r])["u"]
    T5 = rs.pair_t(flat(lx, r), u5, d["cl"][r])
    years.append({"year": y, "market lead t": g["t"][0, 2], "T big → small": T5[4, 0],
                  "T small → big": T5[0, 4]})
longrun = pd.DataFrame(years).set_index("year")
fig = go.Figure()
for j, col in enumerate(longrun.columns):
    fig.add_trace(go.Scatter(
        x=longrun.index, y=longrun[col], mode="lines+markers", name=col,
        line={"color": plots.SERIES[j], "width": 2}, marker={"size": 8},
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>",
    ))
for level in (-2, 2):
    fig.add_hline(y=level, line={"color": plots._MUTED, "dash": "dot", "width": 1})
fig.update_yaxes(title="t / T (week-clustered)")
plots._base(fig, title=f"Fig 7b · daily, {len(deep)} deep names — does the lead hold year "
                       "after year?", height=420)
below(fig).show()

## 8 · Tradability (C6) — validation half, everything fitted on discovery

A lead-lag that exists but cannot be traded is a curiosity. Two trades are tested. Each is decided
at a bar's close, entered at the **next bar's open** (the first print after the signal), held
`hold` bars and never past the close. Each is then repeated **one bar late**, as a latency check.

* **a · market lag.** When the fast market's last `L` bars are among its 10% largest moves
  (threshold from discovery), go long the **slowest** tenth of names (highest discovery lag t) and
  short the **fastest** tenth, in the direction of the market's move. Net of **two** round trips.
* **b · pair network.** Take each follower's **3 strongest discovery leaders**, signed by their
  discovery T. Trade the follower in the predicted direction whenever the prediction's size is in
  its top 1% on discovery. Net of one round trip.

At 1d, the part of the follower's response that happens **overnight** (close → next open) cannot
be captured by a next-open trade; it is reported separately.

In [ ]:
trade, deciles = [], {}
for bar in BARS:
    d, sp = D[bar], SPEC[bar]
    c, n, disc, val = d["c"], d["n"], d["disc"], d["val"]
    clv = d["cl"][val]
    fwd = {dl: rs.forward(c["x"][..., :n], c["y"][..., :n], sp["hold"], delay=dl) for dl in (1, 2)}
    f1d = flat(fwd[1], disc)
    row = {"bar": bar, "round trip (bp)": 1e4 * rt[bar]}
    # a · market lag: slowest minus fastest tenth after a big fast-market move
    lag_d = d["lag_t_disc"]
    lo, hi = np.nanquantile(lag_d, [0.1, 0.9])
    slow, fast = lag_d >= hi, lag_d <= lo
    sig = rs.lagged(d["m"], 0, sp["L"] - 1)
    sd, sv = flat(sig, disc), flat(sig, val)
    thr = np.nanquantile(np.abs(sd[np.isfinite(f1d).any(axis=1)]), 0.9)
    ev = np.flatnonzero(np.abs(sv) >= thr)
    for dl in (1, 2):
        fv = flat(fwd[dl], val)[ev]
        spread = np.nanmean(fv[:, slow], axis=1) - np.nanmean(fv[:, fast], axis=1)
        gross = np.sign(sv[ev]) * spread
        res = rs.day_t(gross - 2 * rt[bar], clv[ev])
        row.update({f"a: net bp, delay {dl}": 1e4 * res["mean"], f"a: t, delay {dl}": res["t"],
                    f"a: trades, delay {dl}": res["n"]})
        if dl == 1:
            row["a: gross bp"] = 1e4 * np.nanmean(gross)
    # b · pair network: each follower's 3 strongest discovery leaders, signed
    Td = d["Td"]
    lead3 = np.argsort(np.nan_to_num(np.abs(Td), nan=-1.0), axis=0)[-3:]
    w = np.sign(np.nan_to_num(np.take_along_axis(Td, lead3, axis=0)))
    e_now = flat(rs.lagged(d["e"], 0, sp["L"] - 1))
    pred = np.zeros_like(e_now)
    for k in range(3):
        pred += w[k] * e_now[:, lead3[k]]
    del e_now
    pdisc, pval = pred[disc], pred[val]
    thr_b = np.nanquantile(np.abs(pdisc[np.isfinite(f1d)]), 0.99)
    for dl in (1, 2):
        fv = flat(fwd[dl], val)
        r_, c_ = np.nonzero((np.abs(pval) >= thr_b) & np.isfinite(fv))
        res = rs.day_t(np.sign(pval[r_, c_]) * fv[r_, c_] - rt[bar], clv[r_])
        row.update({f"b: net bp, delay {dl}": 1e4 * res["mean"], f"b: t, delay {dl}": res["t"],
                    f"b: trades, delay {dl}": res["n"]})
        if dl == 1:
            row["b: gross bp"] = 1e4 * (res["mean"] + rt[bar])
    qs = rs.spread(pval, flat(fwd[1], val), clv)
    row.update({"b: rank IC": qs["ic"], "b: IC t": qs["ic_t"],
                "b: top − bottom decile bp": 1e4 * qs["spread"]})
    deciles[bar] = qs["deciles"]
    if KIND[bar] == "daily":  # the overnight part a next-open trade cannot capture
        gap = flat(c["gap"][..., :n])
        g1 = gap[np.r_[np.arange(1, len(gap)), len(gap) - 1]][val]
        on_a = np.sign(sv[ev]) * (np.nanmean(g1[ev][:, slow], 1) - np.nanmean(g1[ev][:, fast], 1))
        r_, c_ = np.nonzero((np.abs(pval) >= thr_b) & np.isfinite(g1))
        row.update({"a: overnight bp (not capturable)": 1e4 * np.nanmean(on_a),
                    "b: overnight bp (not capturable)":
                        1e4 * np.nanmean(np.sign(pval[r_, c_]) * g1[r_, c_])})
    trade.append(row)
    del pred, fwd
trade = pd.DataFrame(trade).set_index("bar")
display(trade.T.round(3))

fig = trio()
for k, bar in enumerate(BARS, start=1):
    fig.add_trace(go.Bar(
        x=list(range(1, 11)), y=1e4 * deciles[bar], name="mean forward return",
        marker={"color": plots.SERIES[0]}, showlegend=False,
        hovertemplate="decile %{x}<br>%{y:.2f} bp<extra></extra>",
    ), row=1, col=k)
    for s in (1, -1):
        fig.add_hline(y=s * 1e4 * rt[bar], line={"color": plots._INK, "dash": "dot", "width": 1},
                      row=1, col=k)
    fig.update_xaxes(title="prediction decile (validation)", row=1, col=k)
fig.update_yaxes(title="mean next-open forward return (bp)", row=1, col=1)
plots._base(fig, title="Fig 8 · the pair network's predictions by decile (dotted: ± a round trip)",
            height=420)
fig.show()

## 9 · Verdict

Every pre-registered check, per bar size (README "Pre-registration").

**GO** = V1 ∧ [(C1 ∧ C3m ∧ C6a) ∨ (V2 ∧ C2 ∧ C3p ∧ C6b)]: a channel must exist, persist and pay,
all on that same channel. C4 and C5 describe what drives any effect; they never decide the GO.

The CHECKS table below it covers the notebook's own integrity:
- are the rolled nulls calibrated;
- does the canary agree with the power table (where they disagree, trust the canary — it is the
  empirical one);
- were the windows and the links file stamped.

In [ ]:
def mark(ok, applicable=True):
    return "n/a" if not applicable else ("PASS" if ok else "FAIL")


verdict, values, checks = {}, {}, []
for bar in BARS:
    d = D[bar]
    pm, mk, pe = pairmap.loc[bar], market.loc[bar], persist.loc[bar]
    sz, lk, tr = size.loc[bar], links.loc[bar], trade.loc[bar]
    lo, hi = TH["null_std"]
    v1 = bool(lo <= pm["roll T std min"] and pm["roll T std max"] <= hi
              and pm["roll tail share max"] <= TH["null_tail"])
    v2 = bool(pm["canary /20"] >= TH["canary"])
    c1 = bool(mk["C1: pooled lag t"] >= TH["market_t"])
    c2 = bool(v2 and pm["beyond market ≥ 4.5"] > pm["roll max"] and pm["FDR"] <= TH["fdr"]
              and pm["busiest follower share"] <= TH["busiest"])
    c3m = bool(pe["lag t Spearman disc → val"] >= TH["lag_rho"]
               and pe["pooled lag t (validation)"] >= TH["val_t"])
    c3p = bool(v2 and pe["top-100 mean signed T (val)"] >= TH["top_val_t"]
               and pe["top-100 same sign"] >= TH["top_sign"]
               and pe["top-100 mean signed T (val)"] > pe["null 95% mean T"]
               and pe["top-100 same sign"] > pe["null 95% same sign"])
    c4 = bool(sz["Spearman(lag t, log cap)"] <= TH["size_rho"]
              and sz["cap t | stale, price"] <= TH["size_t"]
              and sz["T big → small"] >= TH["big_small"] and sz["T small → big"] <= TH["small_big"])
    c5a = bool(lk["industry: permutation pct"] >= TH["industry_pct"])
    c5b = bool(lk["Hou t"] >= TH["hou_t"])
    c5c = bool(KIND[bar] == "daily" and any(
        lk.get(f"links: mean pct {mp}", np.nan) >= TH["link_pct"]
        and lk.get(f"links: p {mp}", np.nan) <= TH["link_p"]
        and lk.get(f"links: mean pct {mp}", np.nan) > lk.get(f"links: reverse pct {mp}", np.nan)
        for mp in ("fast", "slow")))
    c6a = bool(tr["a: net bp, delay 1"] > 0 and tr["a: t, delay 1"] >= TH["trade_t"]
               and tr["a: net bp, delay 2"] > 0)
    c6b = bool(v2 and tr["b: net bp, delay 1"] > 0 and tr["b: t, delay 1"] >= TH["trade_t"]
               and tr["b: net bp, delay 2"] > 0)
    go_ = v1 and ((c1 and c3m and c6a) or (c2 and c3p and c6b))
    verdict[bar] = {
        "V1 rolled nulls calibrated": mark(v1), "V2 canary (power)": mark(v2),
        "C1 the market leads": mark(c1), "C2 pairs beyond the market": mark(c2, v2),
        "C3m the market lead persists": mark(c3m), "C3p the pairs persist": mark(c3p, v2),
        "C4 size (descriptive)": mark(c4), "C5a industries (descriptive)": mark(c5a),
        "C5b Hou (descriptive)": mark(c5b),
        "C5c links (descriptive)": mark(c5c, KIND[bar] == "daily"),
        "C6a market lag tradable": mark(c6a), "C6b pair network tradable": mark(c6b, v2),
        "GO": "GO" if go_ else "NO-GO",
    }
    values[bar] = {k: v for tbl in (market, pairmap, size, links, persist, trade)
                   for k, v in tbl.loc[bar].items()}
    checks.append((f"{bar}: rolled nulls look like a t",
                   f"std {pm['roll T std min']:.2f}..{pm['roll T std max']:.2f}, "
                   f"tail {pm['roll tail share max']:.1e}", "std 0.85-1.20, tail ≤ 3.4e-5", v1))
    checks.append((f"{bar}: canary agrees with the power table",
                   f"{d['canary']}/20, informative={d['informative']}", "both yes or both no",
                   (d["canary"] >= TH["canary"]) == d["informative"]))
checks += [
    ("windows stamped", f"intraday to {W['intraday']['end']}, daily to {W['daily']['end']}",
     "in the SCORECARD", True),
    ("links.yaml frozen", LINKS_SHA1[:12], "sha1 in the SCORECARD", True),
    ("1m pool size", len(pool), "≥ 400 on a full run", SUBSET is not None or len(pool) >= 400),
]
display(pd.DataFrame(verdict))
display(rs.checks_table(checks))
rs.scorecard({
    "notebook": "lead_lag_explore", "subset": SUBSET, "windows": W, "first_oos": rs.FIRST_OOS,
    "links_sha1": LINKS_SHA1, "names": {b: D[b]["n"] for b in BARS}, "verdict": verdict,
    "values": values, "runtime_s": time.time() - T_START,
    "peak_rss_gb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9,
})